In [4]:
import numpy as np
import pandas as pd
import re
import urllib.request
import zipfile
import os
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import mean_squared_error, f1_score, hamming_loss, jaccard_score

import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [5]:
df_raw = pd.read_csv("movies.csv")

# keep only required columns (dataset uses 'genres' and 'vote_average')
df = df_raw[["genres", "keywords", "tagline", "overview", "vote_average"]].copy()
df = df.dropna(subset=["overview", "vote_average"])
df = df.reset_index(drop=True)

print(df.shape)
df.head(3)

(4800, 5)


,genres,keywords,tagline,overview,vote_average
0,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,Enter the World of Pandora.,"In the 22nd century, a paraplegic Marine is di...",7.2
1,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,"At the end of the world, the adventure begins.","Captain Barbossa, long believed to be dead, ha...",6.9
2,Action Adventure Crime,spy based on novel secret agent sequel mi6,A Plan No One Escapes,A cryptic message from Bond’s past sends him o...,6.3


In [6]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)       # remove URLs
    text = re.sub(r"[^a-z\s]", "", text)      # remove punctuation & numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["overview_clean"]  = df["overview"].apply(preprocess_text)
df["tagline_clean"]   = df["tagline"].fillna("").apply(preprocess_text)
df["keywords_clean"]  = df["keywords"].fillna("").apply(preprocess_text)

df[["overview_clean", "tagline_clean", "keywords_clean"]].head(3)

,overview_clean,tagline_clean,keywords_clean
0,in the nd century a paraplegic marine is dispa...,enter the world of pandora,culture clash future space war space colony so...
1,captain barbossa long believed to be dead has ...,at the end of the world the adventure begins,ocean drug abuse exotic island east india trad...
2,a cryptic message from bonds past sends him on...,a plan no one escapes,spy based on novel secret agent sequel mi


In [7]:
# genres column is space-separated e.g. "Action Adventure Fantasy"
df["genre_list"] = df["genres"].fillna("").apply(lambda x: x.split())

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df["genre_list"])
genre_df     = pd.DataFrame(genre_matrix, columns=mlb.classes_)

print("Total genres:", len(mlb.classes_))
print(mlb.classes_)

Total genres: 22
['Action' 'Adventure' 'Animation' 'Comedy' 'Crime' 'Documentary' 'Drama'
 'Family' 'Fantasy' 'Fiction' 'Foreign' 'History' 'Horror' 'Movie' 'Music'
 'Mystery' 'Romance' 'Science' 'TV' 'Thriller' 'War' 'Western']


In [8]:
# indices split: 70 / 15 / 15
idx_train_val, idx_test = train_test_split(df.index, test_size=0.15, random_state=SEED)
idx_train, idx_val      = train_test_split(idx_train_val, test_size=0.1765, random_state=SEED)
# 0.1765 of 85% ≈ 15% of full data

print(f"Train: {len(idx_train)}  Val: {len(idx_val)}  Test: {len(idx_test)}")

Train: 3359  Val: 721  Test: 720


In [9]:
GLOVE_DIR  = "glove"
GLOVE_FILE = os.path.join(GLOVE_DIR, "glove.6B.100d.txt")
GLOVE_URL  = "http://nlp.stanford.edu/data/glove.6B.zip"
GLOVE_ZIP  = "glove.6B.zip"

if not os.path.exists(GLOVE_FILE):
    os.makedirs(GLOVE_DIR, exist_ok=True)
    print("Downloading GloVe 6B zip (~822 MB)...")
    urllib.request.urlretrieve(GLOVE_URL, GLOVE_ZIP)
    with zipfile.ZipFile(GLOVE_ZIP, "r") as zf:
        zf.extract("glove.6B.100d.txt", GLOVE_DIR)
    print("Done.")

glove_vectors = {}
with open(GLOVE_FILE, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.split()
        word  = parts[0]
        vec   = np.array(parts[1:], dtype=np.float32)
        glove_vectors[word] = vec

print(f"GloVe vocab size: {len(glove_vectors):,}  |  Dim: 100")

Done.
GloVe vocab size: 400,000  |  Dim: 100


In [10]:
GLOVE_DIR  = "glove"
GLOVE_FILE = os.path.join(GLOVE_DIR, "glove.6B.100d.txt")
GLOVE_URL  = "http://nlp.stanford.edu/data/glove.6B.zip"
GLOVE_ZIP  = "glove.6B.zip"

if not os.path.exists(GLOVE_FILE):
    os.makedirs(GLOVE_DIR, exist_ok=True)
    print("Downloading GloVe 6B zip (~822 MB)...")
    urllib.request.urlretrieve(GLOVE_URL, GLOVE_ZIP)
    with zipfile.ZipFile(GLOVE_ZIP, "r") as zf:
        zf.extract("glove.6B.100d.txt", GLOVE_DIR)
    print("Done.")

glove_vectors = {}
with open(GLOVE_FILE, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.split()
        word  = parts[0]
        vec   = np.array(parts[1:], dtype=np.float32)
        glove_vectors[word] = vec

print(f"GloVe vocab size: {len(glove_vectors):,}  |  Dim: 100")

GloVe vocab size: 400,000  |  Dim: 100


In [11]:
all_tokens = " ".join(df["overview_clean"] + " " + df["tagline_clean"] + " " + df["keywords_clean"])
unique_tokens = set(all_tokens.split())

covered = unique_tokens & set(glove_vectors.keys())
coverage_pct = len(covered) / len(unique_tokens) * 100

print(f"Unique dataset tokens : {len(unique_tokens):,}")
print(f"Covered by GloVe      : {len(covered):,}")
print(f"Coverage              : {coverage_pct:.2f}%")

Unique dataset tokens : 24,678
Covered by GloVe      : 21,750
Coverage              : 88.14%


In [12]:
EMBED_DIM = 100

def build_tfidf_glove_embeddings(text_series, glove, dim=EMBED_DIM):
    tfidf = TfidfVectorizer(max_features=20000)
    tfidf.fit(text_series)
    vocab_idf = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

    embeddings = []
    for text in text_series:
        tokens = text.split()
        weighted_vecs = []
        for token in tokens:
            if token in glove and token in vocab_idf:
                weighted_vecs.append(glove[token] * vocab_idf[token])
        if weighted_vecs:
            doc_vec = np.mean(weighted_vecs, axis=0)
        else:
            doc_vec = np.zeros(dim)
        embeddings.append(doc_vec)

    return np.array(embeddings, dtype=np.float32)

# Build for all three text columns
emb_overview  = build_tfidf_glove_embeddings(df["overview_clean"],  glove_vectors)
emb_tagline   = build_tfidf_glove_embeddings(df["tagline_clean"],   glove_vectors)
emb_keywords  = build_tfidf_glove_embeddings(df["keywords_clean"],  glove_vectors)

print("Embedding shapes:", emb_overview.shape, emb_tagline.shape, emb_keywords.shape)

Embedding shapes: (4800, 100) (4800, 100) (4800, 100)


In [13]:
class RatingRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

In [14]:
def train_and_eval_regressor(embeddings, ratings, idx_train, idx_val, idx_test, epochs=50, lr=1e-3):
    X_train = torch.tensor(embeddings[idx_train])
    X_val   = torch.tensor(embeddings[idx_val])
    X_test  = torch.tensor(embeddings[idx_test])

    y_train = torch.tensor(ratings[idx_train], dtype=torch.float32)
    y_val   = torch.tensor(ratings[idx_val],   dtype=torch.float32)
    y_test  = torch.tensor(ratings[idx_test],  dtype=torch.float32)

    model     = RatingRegressor(input_dim=EMBED_DIM)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        preds = model(X_train)
        loss  = loss_fn(preds, y_train)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        test_preds = model(X_test).numpy()
        y_test_np  = y_test.numpy()

    mse  = mean_squared_error(y_test_np, test_preds)
    rmse = np.sqrt(mse)
    return mse, rmse

ratings_array = df["vote_average"].values

In [15]:
# Baseline: always predict global mean
global_mean  = ratings_array[idx_train].mean()
baseline_mse = mean_squared_error(ratings_array[idx_test], np.full(len(idx_test), global_mean))
print(f"Baseline (mean pred) MSE: {baseline_mse:.4f}  RMSE: {np.sqrt(baseline_mse):.4f}")

# Overview
mse_ov, rmse_ov = train_and_eval_regressor(emb_overview, ratings_array, idx_train, idx_val, idx_test)
print(f"Overview  → MSE: {mse_ov:.4f}  RMSE: {rmse_ov:.4f}")

# Tagline
mse_tl, rmse_tl = train_and_eval_regressor(emb_tagline, ratings_array, idx_train, idx_val, idx_test)
print(f"Tagline   → MSE: {mse_tl:.4f}  RMSE: {rmse_tl:.4f}")

# Keywords
mse_kw, rmse_kw = train_and_eval_regressor(emb_keywords, ratings_array, idx_train, idx_val, idx_test)
print(f"Keywords  → MSE: {mse_kw:.4f}  RMSE: {rmse_kw:.4f}")

Baseline (mean pred) MSE: 1.1490  RMSE: 1.0719
Overview  → MSE: 1.4533  RMSE: 1.2055
Tagline   → MSE: 6.8609  RMSE: 2.6193
Keywords  → MSE: 3.9934  RMSE: 1.9983


In [16]:
class GenreClassifier(nn.Module):
    def __init__(self, input_dim, num_labels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_labels)
        )

    def forward(self, x):
        return self.net(x)   # raw logits → sigmoid applied in loss

In [17]:
def train_and_eval_classifier(embeddings, labels, idx_train, idx_val, idx_test, epochs=60, lr=1e-3, threshold=0.5):
    num_labels = labels.shape[1]

    X_train = torch.tensor(embeddings[idx_train])
    X_test  = torch.tensor(embeddings[idx_test])
    y_train = torch.tensor(labels[idx_train], dtype=torch.float32)
    y_test  = labels[idx_test]

    model     = GenreClassifier(input_dim=EMBED_DIM, num_labels=num_labels)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train)
        loss   = loss_fn(logits, y_train)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        test_logits = model(X_test).numpy()
        test_preds  = (test_logits >= threshold).astype(int)

    micro_f1  = f1_score(y_test, test_preds, average="micro",  zero_division=0)
    macro_f1  = f1_score(y_test, test_preds, average="macro",  zero_division=0)
    ham_loss  = hamming_loss(y_test, test_preds)
    jaccard   = jaccard_score(y_test, test_preds, average="samples", zero_division=0)

    return micro_f1, macro_f1, ham_loss, jaccard

In [18]:
# Overview
mf1_ov, maf1_ov, hl_ov, jc_ov = train_and_eval_classifier(emb_overview, genre_matrix, idx_train, idx_val, idx_test)
print(f"Overview  → Micro-F1: {mf1_ov:.4f}  Macro-F1: {maf1_ov:.4f}  Hamming: {hl_ov:.4f}  Jaccard: {jc_ov:.4f}")

# Tagline
mf1_tl, maf1_tl, hl_tl, jc_tl = train_and_eval_classifier(emb_tagline, genre_matrix, idx_train, idx_val, idx_test)
print(f"Tagline   → Micro-F1: {mf1_tl:.4f}  Macro-F1: {maf1_tl:.4f}  Hamming: {hl_tl:.4f}  Jaccard: {jc_tl:.4f}")

# Keywords
mf1_kw, maf1_kw, hl_kw, jc_kw = train_and_eval_classifier(emb_keywords, genre_matrix, idx_train, idx_val, idx_test)
print(f"Keywords  → Micro-F1: {mf1_kw:.4f}  Macro-F1: {maf1_kw:.4f}  Hamming: {hl_kw:.4f}  Jaccard: {jc_kw:.4f}")

Overview  → Micro-F1: 0.1490  Macro-F1: 0.0396  Hamming: 0.1125  Jaccard: 0.1056
Tagline   → Micro-F1: 0.0675  Macro-F1: 0.0205  Hamming: 0.1186  Jaccard: 0.0428
Keywords  → Micro-F1: 0.2921  Macro-F1: 0.1295  Hamming: 0.1062  Jaccard: 0.1988


In [19]:
STOPWORDS = set(["the","a","an","and","of","to","in","is","it","that","with",
                 "for","on","as","this","by","are","was","be","at","from","or"])

def get_word_freq_per_genre(text_col, genre_col, min_freq=3, top_n=10):
    results = {}
    all_genres = sorted(set(g for gl in genre_col for g in gl))

    for genre in all_genres:
        mask  = genre_col.apply(lambda gl: genre in gl)
        words = " ".join(text_col[mask]).split()
        words = [w for w in words if w not in STOPWORDS and len(w) > 2]
        freq  = Counter(words)

        top10    = freq.most_common(top_n)
        eligible = {w: c for w, c in freq.items() if c >= min_freq}
        bot10    = sorted(eligible.items(), key=lambda x: x[1])[:top_n]

        results[genre] = {"top10": top10, "bottom10": bot10}

    return results

word_freq = get_word_freq_per_genre(df["overview_clean"], df["genre_list"])

for genre, data in list(word_freq.items())[:5]:   # print first 5 genres
    print(f"\n=== {genre} ===")
    print("Top 10   :", [w for w, _ in data["top10"]])
    print("Bottom 10:", [w for w, _ in data["bottom10"]])


=== Action ===
Top 10   : ['his', 'when', 'who', 'their', 'they', 'but', 'has', 'her', 'him', 'into']
Bottom 10: ['believed', 'turner', 'message', 'bonds', 'spectre', 'dent', 'avengers', 'alliances', 'modernday', 'sort']

=== Adventure ===
Top 10   : ['his', 'their', 'when', 'who', 'they', 'but', 'her', 'has', 'world', 'into']
Bottom 10: ['dispatched', 'unique', 'orders', 'protecting', 'believed', 'message', 'bonds', 'spectre', 'transported', 'embroiled']

=== Animation ===
Top 10   : ['his', 'when', 'their', 'but', 'her', 'who', 'world', 'they', 'has', 'new']
Bottom 10: ['looking', 'duo', 'overprotective', 'look', 'relationship', 'mcqueen', 'international', 'andy', 'left', 'nefarious']

=== Comedy ===
Top 10   : ['his', 'her', 'their', 'when', 'who', 'they', 'but', 'has', 'she', 'him']
Bottom 10: ['mcqueen', 'mater', 'espionage', 'havent', 'scottish', 'mrida', 'unruly', 'accomplished', 'enormous', 'leaders']

=== Crime ===
Top 10   : ['his', 'when', 'who', 'her', 'their', 'but', 'the

In [20]:
tfidf_indicative = TfidfVectorizer(max_features=15000, stop_words="english")
tfidf_matrix     = tfidf_indicative.fit_transform(df["overview_clean"])
feature_names    = tfidf_indicative.get_feature_names_out()

print(f"{'Genre':<25} {'Top 10 Indicative Words'}")
print("-" * 75)

for i, genre in enumerate(mlb.classes_):
    genre_labels = genre_matrix[:, i]

    if genre_labels.sum() < 10:   # skip very rare genres
        continue

    lr = LogisticRegression(max_iter=500, C=1.0, solver="lbfgs")
    lr.fit(tfidf_matrix, genre_labels)

    top_idx   = np.argsort(lr.coef_[0])[-10:][::-1]
    top_words = [feature_names[j] for j in top_idx]

    print(f"{genre:<25} {top_words}")

Genre                     Top 10 Indicative Words
---------------------------------------------------------------------------
Action                    ['agent', 'cop', 'criminals', 'ruthless', 'mission', 'hero', 'cia', 'kidnapped', 'target', 'forces']
Adventure                 ['adventure', 'bond', 'world', 'mission', 'earth', 'save', 'captain', 'evil', 'quest', 'king']
Animation                 ['adventure', 'animated', 'world', 'save', 'shrek', 'named', 'human', 'animals', 'dragon', 'journey']
Comedy                    ['comedy', 'big', 'wedding', 'movie', 'guy', 'friends', 'doesnt', 'christmas', 'comic', 'single']
Crime                     ['police', 'cop', 'murder', 'drug', 'criminal', 'detective', 'fbi', 'mafia', 'mob', 'crime']
Documentary               ['documentary', 'look', 'film', 'interviews', 'footage', 'filmmaker', 'michael', 'fans', 'filmmakers', 'global']
Drama                     ['story', 'life', 'wife', 'drama', 'father', 'war', 'family', 'love', 'lives', 'mother']
F

In [21]:
summary = pd.DataFrame({
    "Input Column": ["Overview", "Tagline", "Keywords"],
    "Reg MSE":      [round(mse_ov,4), round(mse_tl,4), round(mse_kw,4)],
    "Reg RMSE":     [round(rmse_ov,4), round(rmse_tl,4), round(rmse_kw,4)],
    "Micro F1":     [round(mf1_ov,4), round(mf1_tl,4), round(mf1_kw,4)],
    "Macro F1":     [round(maf1_ov,4), round(maf1_tl,4), round(maf1_kw,4)],
    "Hamming Loss": [round(hl_ov,4), round(hl_tl,4), round(hl_kw,4)],
    "Jaccard":      [round(jc_ov,4), round(jc_tl,4), round(jc_kw,4)],
})

print(summary.to_string(index=False))

Input Column  Reg MSE  Reg RMSE  Micro F1  Macro F1  Hamming Loss  Jaccard
    Overview   1.4533    1.2055    0.1490    0.0396        0.1125   0.1056
     Tagline   6.8609    2.6193    0.0675    0.0205        0.1186   0.0428
    Keywords   3.9934    1.9983    0.2921    0.1295        0.1062   0.1988
